# OLMo-2-1B OpenMath Tanya-style polar sweep

Repo-native OpenMathInstruct-2 comparison inspired by `tanya_results/owm300m_polar_sweep.md`. Frank-Wolfe is chord-tight-clean k=1; BCD is chord-tight-clean k=2; KL-diag+polar uses cw_picard_iters=1. Each arm has three LRs and no damping grid; AdamW uses weight_decay=0.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lora_playground').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure

GROUP = 'openmath_tanya_style_polar_r64'
HORIZON = 9000
VARIANTS = {
    'AdamW': {},
    'Muon': {},
    'Frank-Wolfe (clean k=1)': {},
    'BCD (clean k=2)': {},
    'KL-diag + polar': {},
}

def effective_picard(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def method_label(cfg):
    opt = cfg.get('optimizer')
    if opt == 'adamw':
        return 'AdamW'
    if opt == 'muon-lora':
        return 'Muon'
    if opt == 'kl-diag-polar-lora':
        return 'KL-diag + polar'
    if opt == 'adam-polar-product-lora-coupled-spectral-chord-tight-clean':
        k = int(effective_picard(cfg))
        if k == 1:
            return 'Frank-Wolfe (clean k=1)'
        if k == 2:
            return 'BCD (clean k=2)'
    return None


In [ ]:
runs = [
    (cfg, hist)
    for cfg, hist in load_runs(where={'_group': GROUP})
    if hist and method_label(cfg) in VARIANTS
]

fig, table_df, summary_df = compare_variants_figure(
    VARIANTS,
    common_where={},
    ref_label='AdamW',
    target_label='AdamW',
    prefetched_runs=runs,
    variant_key=method_label,
    max_steps=HORIZON,
    allow_partial=True,
    allow_custom_labels=True,
    suptitle='OLMo-2-1B OpenMath r=64 Tanya-style polar sweep',
    figsize=(12, 4.5),
)
plt.show()
